# Advection-diffusion of a Gaussian in a rectangle

In [1]:
import numpy as np
from lucifex.mesh import rectangle_mesh, mesh_boundary
from lucifex.fdm import (
    CN, AM1, AB1, AB2, FiniteDifference, FiniteDifferenceArgwise,
    FunctionSeries, ConstantSeries, advective_diffusive_timestep,
)
from lucifex.fem import Constant, Function, cross_section_grid
from lucifex.solver import ibvp, BoundaryConditions, evaluation
from lucifex.sim import Simulation, run
from lucifex.plt import plot_colormap, plot_line, save_figure, create_animation, display_animation
from lucifex.pde.advection_diffusion import advection_diffusion

def velocity(
    t: Constant | float,
    aMax: float,
    aFreq: float,
) -> tuple[float, float]:
    return (aMax * np.sin(aFreq * float(t)), aMax * np.cos(aFreq * float(t)))

def create_simulation(
    Lx: float,
    Ly: float,
    Nx: int,
    Ny: int,
    dt: float,
    aMax: float,
    aFreq: float,
    D_adv: FiniteDifference | FiniteDifferenceArgwise,
    D_diff: FiniteDifference,
) -> Simulation:
    mesh = rectangle_mesh(Lx, Ly, Nx, Ny)
    boundary = mesh_boundary(
        mesh, 
        {
            "left": lambda x: x[0],
            "right": lambda x: x[0] - Lx,
            "lower": lambda x: x[1],
            "upper": lambda x: x[1] - Ly,
        },
    )
    t = ConstantSeries(mesh, name='t', ics=0.0)
    dt = Constant(mesh, dt, name='dt')
    a = FunctionSeries((mesh, 'P', 1, 2), name='a')
    d = Constant(mesh, 0.1, name='d')
    a_solver = evaluation(a, velocity, future=False)(t[0], aMax, aFreq)
    ics = lambda x: np.exp(-((x[0] - Lx/2)**2 + (x[1] - Ly/2)**2)/ (0.01 * Lx))
    bcs = BoundaryConditions(
        ("dirichlet", boundary.union, 0.0),  
    )
    u = FunctionSeries((mesh, 'P', 1), name='u', store=1)
    u_solver = ibvp(advection_diffusion, ics, bcs)(u, dt, a, d, D_adv, D_diff)
    solvers = [a_solver, u_solver]
    return Simulation(solvers, t, dt)


Nx = 100
Ny = 100
Lx = 2.0
Ly = 1.0

D_adv_opts = (AB1, AB1 @ CN, AB2 @ CN)